In [1]:
import polars as pl
from procompa import get_project_root
import matplotlib.pyplot as plt
import py3Dmol
import gemmi
import numpy as np
import py3Dmol
from Bio.PDB import PDBParser, Superimposer, PDBIO

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

In [ ]:
# File stays on the cluster — only coordinates are sent to the browser
with open("/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pool_0.pdb", "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "blue"}})          # PyMOL-like cartoon
view.setStyle({"hetflag": True}, {"stick": {}})            # ligands as sticks
view.setStyle({"bonds": 0}, {"sphere": {"radius": 0.5}})   # ions/waters as spheres
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
ref_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pair_0.pdb"
mob_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pool_0.pdb"
aligned_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/pool_aligned.pdb"

parser = PDBParser(QUIET=True)
ref = parser.get_structure("ref", ref_path)
mob = parser.get_structure("mob", mob_path)

# --- Match CA atoms by (chain, resseq) ---
ref_ca = {(a.get_parent().get_parent().id, a.get_parent().id[1]): a
          for a in ref.get_atoms() if a.name == "CA"}
mob_ca = {(a.get_parent().get_parent().id, a.get_parent().id[1]): a
          for a in mob.get_atoms() if a.name == "CA"}

common = sorted(set(ref_ca.keys()) & set(mob_ca.keys()))
ref_atoms = [ref_ca[r] for r in common]
mob_atoms = [mob_ca[r] for r in common]

print(f"Matched {len(common)} CA atoms")
print(f"Ref chains: {sorted(set(k[0] for k in ref_ca))}")
print(f"Mob chains: {sorted(set(k[0] for k in mob_ca))}")

# --- Iterative outlier rejection (mimics PyMOL align) ---
def align_with_rejection(ref_atoms, mob_atoms, max_cycles=5, reject_factor=2.0):
    ref_list = list(ref_atoms)
    mob_list = list(mob_atoms)
    sup = Superimposer()

    for cycle in range(max_cycles):
        sup.set_atoms(ref_list, mob_list)
        sup.apply(mob_list)

        devs = np.array([np.linalg.norm(r.coord - m.coord)
                         for r, m in zip(ref_list, mob_list)])
        rmsd = sup.rms
        n = len(ref_list)
        threshold = reject_factor * rmsd
        keep = devs < threshold

        print(f"Cycle {cycle+1}: {n} atoms, RMSD={rmsd:.2f} Å, "
              f"threshold={threshold:.1f} Å, rejected={int((~keep).sum())}")

        if keep.all() or n <= 10:
            break
        ref_list = [a for a, k in zip(ref_list, keep) if k]
        mob_list = [a for a, k in zip(mob_list, keep) if k]

    # Apply final transform to ALL atoms in the mobile structure
    sup.set_atoms(ref_list, mob_list)
    sup.apply(list(mob.get_atoms()))
    return sup

sup = align_with_rejection(ref_atoms, mob_atoms)
print(f"Final RMSD: {sup.rms:.2f} Å")


io = PDBIO()
io.set_structure(mob)
io.save(aligned_path)

with open(ref_path) as f:
    ref_data = f.read()
with open(aligned_path) as f:
    mob_data = f.read()

view = py3Dmol.view(width=700, height=500)
view.addModel(ref_data, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "blue"}})
view.addModel(mob_data, "pdb")
view.setStyle({"model": 1}, {"cartoon": {"color": "magenta"}})
view.zoomTo()
view.show()

Matched 722 CA atoms
Ref chains: ['A', 'B']
Mob chains: ['A', 'B']
Cycle 1: 722 atoms, RMSD=41.61 Å, threshold=83.2 Å, rejected=35
Cycle 2: 687 atoms, RMSD=35.32 Å, threshold=70.6 Å, rejected=13
Cycle 3: 674 atoms, RMSD=33.81 Å, threshold=67.6 Å, rejected=11
Cycle 4: 663 atoms, RMSD=32.54 Å, threshold=65.1 Å, rejected=8
Cycle 5: 655 atoms, RMSD=31.50 Å, threshold=63.0 Å, rejected=2
Final RMSD: 31.31 Å


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [2]:

benchmark_structure_similarity = pl.read_parquet(data_dir / "Pipeline/8_benchmark_part_two/cf_pdb_structure_similarity/both_parts_benchmark_structure_similarity.parquet")
benchmark_structure_similarity = benchmark_structure_similarity.filter(pl.col("n_proteins")>2)
benchmark_structure_similarity = benchmark_structure_similarity.with_columns(
    usalign_pdb_path=pl.col("usalign_pred_dir") + "/complex/usalign.pdb"
)

In [24]:
outliers = (
    benchmark_structure_similarity
    .filter((pl.col("CF_confidence") > 80) & (pl.col("usalign_cpx_tm_score") < 0.5))
    .select(["complex_ac", "CF_confidence", "usalign_cpx_tm_score"])
    .sort("usalign_cpx_tm_score")
)
print(outliers)

shape: (4, 3)
┌────────────┬───────────────┬──────────────────────┐
│ complex_ac ┆ CF_confidence ┆ usalign_cpx_tm_score │
│ ---        ┆ ---           ┆ ---                  │
│ str        ┆ f64           ┆ f64                  │
╞════════════╪═══════════════╪══════════════════════╡
│ CPX-599    ┆ 85.2862       ┆ 0.27829              │
│ CPX-2566   ┆ 83.125        ┆ 0.44309              │
│ CPX-599    ┆ 85.47         ┆ 0.48593              │
│ CPX-599    ┆ 85.5743       ┆ 0.49753              │
└────────────┴───────────────┴──────────────────────┘


In [4]:
"""
Build a py3Dmol overlay view: reference + CombFold prediction
(colored by chain), already superposed by US-align so no re-alignment here.

Cleaning mirrors the earlier PyMOL script:
  - drop waters/ligands (hetero, non-polymer)
  - drop nucleic acid chains (protein-only comparison)
"""


def clean_structure_to_pdb_string(path: str) -> str:
    st = gemmi.read_structure(path)
    st.setup_entities()
    st.remove_ligands_and_waters()

    for model in st:
        drop = []
        for i, chain in enumerate(model):
            polymer = chain.get_polymer()
            ptype = polymer.check_polymer_type()
            if ptype in (
                gemmi.PolymerType.Dna,
                gemmi.PolymerType.Rna,
                gemmi.PolymerType.DnaRnaHybrid,
            ):
                drop.append(i)
        for i in reversed(drop):
            del model[i]

    st.remove_empty_chains()
    return st.make_pdb_string()


def make_overlay_view(ref_path: str, pred_path: str, width: int = 700, height: int = 500):
    """model 0 = reference (grey ghost), model 1 = prediction (colored by chain)."""
    ref_pdb = clean_structure_to_pdb_string(ref_path)
    pred_pdb = clean_structure_to_pdb_string(pred_path)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(ref_pdb, "pdb")
    view.addModel(pred_pdb, "pdb")

    view.setStyle({"model": 0}, {"cartoon": {"color": "0x0072B2"}})   # reference (PDB) — blue
    view.setStyle({"model": 1}, {"cartoon": {"color": "0xE69F00"}})   # prediction (CF) — orange

    view.zoomTo()
    return view

In [5]:

def show(complex_ac: str, which: str = "best"):
    rows = benchmark_structure_similarity.filter(pl.col("complex_ac") == complex_ac)
    rows = rows.with_columns(
        pred_tag=pl.col("usalign_pred_dir").str.extract(r"(usalign_outputs_pred\d+)$", 1)
    )
    if which == "best":
        row = rows.sort("usalign_cpx_tm_score", descending=True).row(0, named=True)
    else:
        row = rows.filter(pl.col("pred_tag") == which).row(0, named=True)

    print(f"{row['complex_ac']}  [{row['pred_tag']}]  TM={row['usalign_cpx_tm_score']:.2f}  CF={row['CF_confidence']:.0f}")
    print("ref: ", row["reference_pdb_path"])
    print("pred:", row["usalign_pdb_path"])
    view = make_overlay_view(row["reference_pdb_path"], row["usalign_pdb_path"])
    return view.show()

In [6]:
show("CPX-45", which="usalign_outputs_pred0")

CPX-45  [usalign_outputs_pred0]  TM=0.94  CF=84
ref:  /cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb/4c92/4c92-assembly1.cif
pred: /cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/7_benchmark_part_one/CombFold/P38203x1_P40070x1_P40089x1_P47017x1_P53905x1_P57743x1_Q06406x1_pool_output/usalign_outputs/usalign_outputs_pred0/complex/usalign.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.